## Phase 2: Use NuExtract-1.5 to extract phrases

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import re
import os
import math
import json
from datetime import datetime
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import ftfy
import gc


# ---------- 2.0 Configurate ----------
current_dir = os.path.dirname(os.path.abspath("__file__")) if "__file__" in globals() else os.getcwd()
data_file = os.path.join(current_dir, "data", "df_cleaned.csv")
output_dir = os.path.join(current_dir, "output")
os.makedirs(output_dir, exist_ok=True)
trunc_log_path = os.path.join(output_dir, "truncated_rows_log.txt")

model_name = "numind/NuExtract-1.5"
batch_size=1
max_length=2048
max_new_tokens=512
chunk_size = 5  # process 5 rows at a time

final_file = os.path.join(output_dir, "df_final_with_extracted_features.csv")
json_file = os.path.join(output_dir, "raw_outputs.jsonl")

# ---------- 2.1 Load data ----------

df_cleaned = pd.read_csv(data_file)
print("Loaded:", data_file)
print(df_cleaned.head())


# ---------- 2.2 Load NuExtract-1.5 on GPU ----------

print("Loading model......")
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    trust_remote_code=True
).to(device).eval()

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

print("Model and tokenizer loaded.")


# ---------- 2.3 JSON template and keyword lists ----------

template_dict = {
    "luxury_features": "",
    "transport_mentions": "",
    "school_mentions": "",
    "renovation_mentions": ""
}
template_str = json.dumps(template_dict, indent=4)

luxury_keywords = [
    "luxury", "luxurious", "superb", "stunning", "spectacular", "magnificent",
    "exceptional", "impressive", "beautifully presented", "immaculately presented",
    "high specification", "high-specification", "finished to a high standard",
    "bespoke", "interior design", "designer", "state of the art", "state-of-the-art",
    "concierge", "24 hour concierge", "porter", "doorman", "lift", "elevator",
    "gated", "security", "secure", "video entry", "private entrance",
    "gym", "spa", "swimming pool", "pool", "sauna", "steam room",
    "cinema room", "media room", "games room", "wine cellar", "wine room",
    "private terrace", "roof terrace", "terrace", "balcony",
    "staff accommodation", "home automation", "billiards room", "bar",
    "treatment room", "jacuzzi", "tennis court", "gymnasium", "massage room",
    "humidor", "cigar room", "whiskey bar", "library", "study",
    "home cinema", "virtual room", "squash court", "business lounge",
    "temperature-controlled wine cellar", "private courtyard", "internal garden",
    "water feature", "ornamental pond", "360º views",
    "landscaped garden", "private garden", "rear garden", "garden",
    "communal gardens", "river view", "park view", "panoramic views",
    "penthouse", "mayfair", "knightsbridge", "kensington", "chelsea",
    "exclusive", "prestigious", "prime location"
]

transport_keywords = [
    "transport links", "excellent transport links", "good transport links",
    "close to transport", "easy access", "quick access", "moments from",
    "within walking distance",
    "station", "underground", "tube", "overground", "rail", "train",
    "bus", "bus routes", "line", "stone's throw", "close to",
    "within reach", "nearby", "proximity", "central location",
    "well-connected", "well-served", "transport hub", "road access",
    "short walk", "walking distance", "commute", "connections", "accessible",
    "city", "west end", "central london", "minutes away", "short distance"
]

school_keywords = [
    "schools", "school", "excellent schools", "good schools",
    "local schools", "near schools", "close to schools",
    "catchment", "catchment area",
    "primary school", "secondary school",
    "college", "university",
    "academy", "institute", "education", "primary schools", "institutes"
]

renovation_keywords = [
    "renovated", "newly renovated", "recently renovated",
    "refurbished", "newly refurbished", "recently refurbished",
    "modernised", "modernized", "upgraded", "redecorated",
    "refitted", "updated", "brand new", "rebuilt", "reconstructed",
    "remodeled", "redesigned", "reimagined", "renewed", "reconditioned",
    "excellent condition", "good condition", "immaculate condition",
    "turnkey", "move-in ready", "ready to move into",
    "restored", "redeveloped", "reconfigured",
    "completely refurbished", "fully refurbished",
    "finished", "new kitchen", "new bathrooms"
]


def build_prompt(text: str) -> str:
    instructions = f"""
You are extracting structured information from a London real-estate listing.

GENERAL RULES
- Read the ENTIRE listing carefully.
- For each field, return a SHORT list of key phrases, separated by semicolons (;).
- Each phrase must be copied VERBATIM from the text (no paraphrasing).
- Do NOT invent information. If nothing relevant is found for a field, set that field to "" (empty string).
- Avoid full sentences; use compact phrases only.

FIELD DEFINITIONS

1) "luxury_features":
   - Phrases that indicate luxury amenities, services, finishes, or exclusive character.
   - Focus especially on phrases that include or are similar to keywords like:
     {", ".join(luxury_keywords)}
   - Examples of good outputs:
     "indoor swimming pool"; "spa with sauna and steam room"; "24 hour concierge";
     "landscaped private garden"; "home cinema"; "temperature-controlled wine cellar".

2) "transport_mentions":
   - Phrases that describe public transport, road access, or how easy it is to reach key areas.
   - Focus especially on phrases that include or are similar to keywords like:
     {", ".join(transport_keywords)}
   - Examples:
     "short walk to Knightsbridge Underground Station";
     "excellent transport links to the City and the West End";
     "within walking distance of Victoria Station".

3) "school_mentions":
   - Phrases that mention schools, school quality, or proximity to education.
   - Focus especially on phrases that include or are similar to keywords like:
     {", ".join(school_keywords)}
   - Examples:
     "excellent local schools";
     "close to top independent schools";
     "within the catchment area of outstanding primary schools".

4) "renovation_mentions":
   - Phrases that describe renovation, refurbishment, modernisation, or condition of the property.
   - Focus especially on phrases that include or are similar to keywords like:
     {", ".join(renovation_keywords)}
   - Examples:
     "newly refurbished throughout";
     "recently renovated to a high specification";
     "turnkey condition";
     "comprehensively redeveloped".

OUTPUT FORMAT
- Return a single JSON object exactly matching this template:
{template_str}

- Each value must be a single string containing zero or more phrases separated by semicolons.
- Do NOT add extra keys or commentary.
"""
    return f"""<|input|>
{instructions.strip()}

### Text:
{text}

<|output|>"""


# ---------- 2.4 Append the truncated row into log file ----------

def log_truncation(idx, original_len, kept_len):
    with open(trunc_log_path, "a", encoding="utf-8") as f:
        f.write(
            f"[{datetime.now().isoformat()}] "
            f"row_index={idx}, original_chars={original_len}, kept_chars={kept_len}\n"
        )


# ---------- 2.5 Run NuExtract on a batch of texts ----------

def nuextract_batch(texts, row_indices, batch_size, max_length, max_new_tokens):
    device_local = model.device
    prompts = []
    # track which prompts were truncated at character level
    for idx, t in zip(row_indices, texts):
        prompt = build_prompt(t)
        # rough char-length check before tokenization
        if len(prompt) > max_length * 4:  # heuristic: ~4 chars per token
            log_truncation(idx, len(prompt), max_length * 4)
            prompt = prompt[-max_length * 4:]  # keep last part of prompt
        prompts.append(prompt)

    outputs = []

    with torch.no_grad():
        for i in range(0, len(prompts), batch_size):
            batch_prompts = prompts[i:i + batch_size]

            enc = tokenizer(
                batch_prompts,
                return_tensors="pt",
                truncation=True,
                padding=True,
                max_length=max_length
            ).to(device_local)

            pred_ids = model.generate(
                **enc,
                max_new_tokens=max_new_tokens,  
                do_sample=False,
                use_cache=True
            )

            decoded = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)

            for out in decoded:
                if "<|output|>" in out:
                    outputs.append(out.split("<|output|>", 1)[1].strip())
                else:
                    outputs.append(out.strip())
    return outputs


# ---------- 2.6 Robust JSON extraction (one line per input) ----------

empty_obj = {
    "luxury_features": "",
    "transport_mentions": "",
    "school_mentions": "",
    "renovation_mentions": ""
}

def extract_last_json(text: str):

    # Take the model output and: 1) find the last {...} block 2) parse it as JSON 3) ensure all expected keys exist.    
    start = text.rfind("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return empty_obj.copy(), None

    candidate = text[start:end+1]
    # collapse whitespace to keep it one line
    candidate_one_line = re.sub(r"\s+", " ", candidate).strip()

    try:
        obj = json.loads(candidate_one_line)
    except json.JSONDecodeError:
        return empty_obj.copy(), None

    # Ensure keys
    for k in empty_obj.keys():
        obj.setdefault(k, "")

    return obj, candidate_one_line


# ---------- 2.7 Loop over dataset in chunks of chunk_size rows to run model ----------

# 1) Write CSV header once (empty file with header)
if not os.path.exists(final_file):
    sample_df = df_cleaned.iloc[:1, :]
    dummy_extracted = pd.DataFrame([empty_obj])
    dummy_final = pd.concat([sample_df.reset_index(drop=True), dummy_extracted], axis=1)
    dummy_final.iloc[0:0].to_csv(final_file, index=False)  # write only header

# 2) Ensure JSONL file exists but DO NOT clear if you want resume
if not os.path.exists(json_file):
    open(json_file, "w", encoding="utf-8").close()

n_rows = len(df_cleaned)

# Count how many rows were already processed (1 line per row)
with open(json_file, "r", encoding="utf-8") as f:
    processed_count = sum(1 for _ in f)

print("Already processed rows:", processed_count)

resume_from = processed_count  # first row index to process next

for start in range(resume_from, n_rows, chunk_size):  # use n_rows to replace the test no after test 
    end = min(start + chunk_size, n_rows)

    free_mem, total_mem = torch.cuda.mem_get_info()
    print(f"GPU Memory Free: {free_mem / 1024**2:.2f} MB")
    print(f"Processing rows {start} to {end - 1}")
    gc.collect()
    torch.cuda.empty_cache()

    df_sample = df_cleaned.iloc[start:end, :]
    texts = df_sample["listingDescription"].fillna("").astype(str).tolist()
    row_indices = df_sample.index.tolist()

    print("Running NuExtract on", len(texts), "descriptions...")
    raw_outputs = nuextract_batch(
        texts,
        row_indices,
        batch_size=batch_size,
        max_length=max_length,
        max_new_tokens=max_new_tokens,
    )
    print("Got outputs:", len(raw_outputs))

    for i, out in enumerate(raw_outputs):
        print(f"RAW {i}:", out)

    # Parse JSON, one line per input
    parsed = []
    clean_json_strings = []

    for out in raw_outputs:
        obj, clean_str = extract_last_json(out)
        parsed.append(obj)
        if clean_str is None:
            clean_str = json.dumps(obj, ensure_ascii=False)
        clean_json_strings.append(clean_str)

    # Append JSONL
    with open(json_file, "a", encoding="utf-8") as f:
        for line in clean_json_strings:
            f.write(line.strip() + "\n")

    print("Appended to JSONL:", json_file)

    # Build and append df_final chunk
    df_extracted = pd.DataFrame(parsed)
    df_final_chunk = pd.concat([df_sample.reset_index(drop=True), df_extracted], axis=1)
    print("df_final_chunk:\n", df_final_chunk)

    df_final_chunk.to_csv(
        final_file,
        mode="a",
        index=False,
        header=False,  # header already written once
    )

    # Free Python & CUDA memory before next chunk
    del raw_outputs, df_extracted, df_final_chunk
    gc.collect()
    torch.cuda.empty_cache()
